# 07. 미니 프로젝트 — 검색 + 요약 에이전트

입문 과정에서 배운 내용을 조합하여, Tavily 웹 검색 도구를 갖춘 리서치 에이전트를 만듭니다.

## 학습 목표

- Tavily 검색 도구를 직접 정의한다
- Deep Agents로 리서치 에이전트를 만든다
- 스트리밍으로 에이전트 실행 과정을 실시간 관찰한다
- LangChain 에이전트로도 같은 작업을 수행하여 비교한다

In [1]:
import sys
!uv pip install --python {sys.executable} "tavily-python>=0.5.0"


Using Python 3.12.10 environment at: c:\Users\User\Desktop\ash\ai-agent-edu\day1\.venv
Checked 1 package in 34ms


## 7.1 환경 설정

이 노트북에는 `TAVILY_API_KEY`가 필요합니다. https://tavily.com 에서 무료로 발급받을 수 있습니다.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요!"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 필요!"

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4")
print("\u2713 환경 준비 완료")

✓ 환경 준비 완료


In [3]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


LangSmith tracing ON — project: day1-labs-dx-ash


In [4]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# import 전에 강제로 환경변수 set
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "day1-labs-test")

# 강제 flush 확인용
from langsmith import Client
client = Client()
print("LangSmith 연결 OK, 프로젝트:", os.environ["LANGCHAIN_PROJECT"])
print("API key tail:", os.environ["LANGCHAIN_API_KEY"][-6:])


LangSmith 연결 OK, 프로젝트: day1-labs-dx-ash
API key tail: 8a666f


## 7.2 검색 도구 정의

Tavily 클라이언트를 래핑하는 검색 함수를 만듭니다.
**docstring**과 **타입 힌트**가 에이전트에 도구 스키마를 알려줍니다.

**도구 함수 작성 규칙:**

`create_deep_agent()`의 `tools` 파라미터에 전달할 검색 함수를 정의합니다. Deep Agents는 함수의 docstring을 도구 설명으로, 타입 힌트를 파라미터 스키마로 자동 변환합니다. 따라서:

- **docstring**: 에이전트가 "이 도구를 언제 사용해야 하는지" 판단하는 근거가 됩니다. 명확하고 구체적으로 작성하세요.
- **타입 힌트**: 에이전트가 올바른 타입의 인자를 전달하도록 합니다. `Literal` 타입을 사용하면 허용 값을 제한할 수 있습니다.
- **Args 섹션**: 각 파라미터의 용도를 설명하면 에이전트가 더 정확하게 인자를 선택합니다.

In [5]:
# import sys
# !{sys.executable} -m pip install tavily-python

In [6]:
from typing import Literal
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 3,
    topic: Literal["general", "news"] = "general",
) -> dict:
    """인터넷에서 정보를 검색합니다.

    Args:
        query: 검색 쿼리
        max_results: 최대 결과 수
        topic: 검색 주제 카테고리
    """
    return tavily.search(query, max_results=max_results, topic=topic)

print("\u2713 검색 도구 준비 완료")

✓ 검색 도구 준비 완료


## 7.3 Deep Agents 리서치 에이전트

`create_deep_agent()`에 검색 도구와 시스템 프롬프트를 전달합니다.

**에이전트의 자동 워크플로:**

에이전트는 사용자의 요청을 받으면 다음과 같은 과정을 자동으로 수행합니다:

1. **계획 수립**: 빌트인 `write_todos` 도구로 작업을 단계별로 분해합니다.
2. **리서치 수행**: 전달된 검색 도구(`internet_search`)를 사용하여 웹에서 정보를 수집합니다.
3. **컨텍스트 관리**: 필요 시 파일 시스템 도구(`write_file`, `read_file`)로 중간 결과를 저장하여 토큰 한도를 관리합니다.
4. **결과 종합**: 수집한 정보를 분석하고 일관된 보고서로 종합합니다.

복잡한 작업의 경우, 에이전트는 전문 서브에이전트를 생성하여 특정 하위 작업의 컨텍스트를 격리할 수도 있습니다.

In [7]:
from deepagents import create_deep_agent

research_agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="당신은 전문 리서처입니다. 웹을 검색한 후 결과를 한국어로 요약하세요.",
)
print("\u2713 리서치 에이전트 생성 완료")

✓ 리서치 에이전트 생성 완료


In [8]:
result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangGraph가 무엇인지 검색해서 3줄로 요약해 주세요."
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

LangGraph는 LangChain이 만든 오픈소스 프레임워크로, LLM 애플리케이션과 AI 에이전트의 흐름을 그래프 구조로 설계·오케스트레이션할 수 있게 해줍니다.  
노드와 엣지 기반으로 상태, 분기, 반복, 도구 호출을 관리해 복잡한 워크플로를 더 체계적으로 구현할 수 있습니다.  
특히 공식 문서 기준으로 durable execution, streaming, human-in-the-loop 같은 기능이 강점입니다.


In [ ]:
yeosu_tour_result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "대한민국 여수시의 관광지와 맛집 동선 연결해서 6시간 코스로 추천 해주세요. 마지막은 여수 EXPO역으로 갈 것입니다."
            }
        ]
    },
    config=lf_config,
)

print(yeosu_tour_result["messages"][-1].content)

여수에서 **6시간 안에 관광지+맛집을 자연스럽게 연결**하고, **마지막 도착을 여수EXPO역**으로 맞추려면 **엑스포역 주변–오동도–아쿠아플라넷–해상케이블카–낭만포차/게장 식사** 동선이 가장 효율적입니다.

## 추천 6시간 코스
**핵심 장점:** 이동이 짧고, 여수 대표 관광 포인트를 무리 없이 묶을 수 있음

### 1) 오동도 산책
- **소요:** 60~80분
- **포인트:** 동백숲길, 방파제, 바다 풍경
- **이유:** EXPO역과 가깝고 여수 대표 산책 코스라 시작점으로 좋습니다.

### 2) 아쿠아플라넷 여수
- **소요:** 70~90분
- **이동:** 오동도 인근
- **포인트:** 실내라 날씨 영향 적음
- **이유:** 오동도와 매우 가까워 연계가 좋습니다.

### 3) 점심 또는 이른 저녁 – 게장백반 / 갈치조림
여수 대표 식사는 보통 아래 2가지가 무난합니다.

#### 선택 A. 게장백반
- **추천 이유:** 여수 대표 음식, 여행 만족도 높음
- **추천 지역:** 봉산동 게장거리 또는 EXPO역 이동 동선상 게장집
- **메뉴 예시:** 간장게장, 양념게장, 새우장, 갓김치

#### 선택 B. 갈치조림
- **추천 이유:** 바다 보면서 먹기 좋고 호불호가 적음
- **추천 지역:** 낭만포차/종포 해양공원 근처 또는 EXPO역 주변

- **소요:** 60분

### 4) 여수 해상케이블카 + 돌산공원
- **소요:** 70~90분
- **포인트:** 여수 바다 전경, 돌산대교 뷰
- **이유:** 여수의 대표적인 “한 컷” 명소입니다.
- **참고:** 케이블카 대기 시간이 길면 돌산공원 전망 위주로 짧게 조정 가능

### 5) 종포해양공원 / 낭만포차 거리 가볍게 산책
- **소요:** 30~40분
- **포인트:** 여수 밤바다 분위기
- **이유:** 사진 찍기 좋고, 마지막 이동 전 마무리 코스로 적합

### 6) 여수EXPO역 이동
- **종착:** 여수EXPO역
- **권장:** 출발 열차

## 7.4 스트리밍으로 과정 관찰

`stream(mode="updates")`로 에이전트가 어떤 단계를 거치는지 실시간으로 확인합니다.

**LangGraph 스트리밍 시스템:**

LangGraph는 완전한 응답이 준비되기 전에 진행 상황을 점진적으로 표시하여 애플리케이션의 반응성을 높이는 포괄적인 스트리밍 시스템을 제공합니다.

| 스트림 모드 | 용도 |
|---|---|
| `values` | 각 그래프 단계 후 **전체 상태**를 스트리밍 |
| `updates` | 각 단계 후 **상태 변경분만** 스트리밍 |
| `messages` | LLM 토큰을 메타데이터와 함께 스트리밍 |
| `custom` | 노드에서 사용자 정의 데이터를 스트리밍 |
| `debug` | 포괄적인 실행 정보를 스트리밍 |

`stream()` (동기) 또는 `astream()` (비동기) 메서드로 스트리밍에 접근하며, 여러 모드를 리스트로 전달하여 동시에 사용할 수도 있습니다. 아래 예제에서는 `updates` 모드를 사용하여 에이전트의 각 단계(도구 호출, 최종 응답)를 실시간으로 출력합니다.

In [9]:
for chunk in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangChain v1의 주요 변경사항을 검색해서 요약해 주세요."
            }
        ]
    },
    stream_mode="updates",
    config=lf_config,
):
    for node_name, node_data in chunk.items():
        if not node_data:
            continue

        msgs = node_data.get("messages", [])

        if hasattr(msgs, "value"):
            msgs = msgs.value

        if not msgs:
            continue

        last = msgs[-1]

        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                print(
                    f"[도구 호출] {tc['name']}({tc['args'].get('query', '')[:50]})"
                )

        elif hasattr(last, "content") and last.content and not hasattr(last, "tool_call_id"):
            content = last.content if isinstance(last.content, str) else str(last.content)

            if content.strip():
                print(f"\n[최종 응답]\n{content}")


[최종 응답]
LangChain v1의 주요 변경사항을 검색해서 요약해 주세요.
[도구 호출] internet_search(LangChain v1 major changes release notes migration)

[최종 응답]
LangChain v1의 주요 변경사항을 요약하면 다음과 같습니다.

### 핵심 변화
- **`create_agent` 도입**
  - v1의 대표적인 새 진입점입니다.
  - 기존 `create_react_agent`보다 더 단순하면서도 확장성이 높습니다.
  - 내부적으로 **LangGraph 기반**으로 동작합니다.

- **미들웨어 중심 구조**
  - 이전의 pre/post model hook 방식이 **middleware**로 통합됐습니다.
  - 동적 프롬프트, 모델 선택, 툴 호출 가로채기, 에러 처리 등을 미들웨어로 구현합니다.
  - 즉, 에이전트 커스터마이징 방식이 더 일관되고 강력해졌습니다.

- **LangGraph 기반 강화**
  - LangChain v1 에이전트는 기본적으로 **LangGraph 위에서 실행**됩니다.
  - 덕분에 상태 관리, 반복 실행, 제어 흐름 확장이 더 쉬워졌습니다.

### API/개발 방식 변경
- **프롬프트 파라미터 이름 변경**
  - Python 기준 기존 prompt 관련 사용 방식이 정리되고, `system_prompt` 중심으로 바뀌었습니다.
  - 동적 프롬프트 삽입은 직접 파라미터보다 미들웨어 사용이 권장됩니다.

- **모델 선택 방식 변경**
  - 미리 바인딩된 모델 방식보다, **미들웨어를 통한 동적 모델 선택**이 중심이 됐습니다.
  - 일부 기존 패턴은 더 이상 지원되지 않습니다.

- **툴 에러 처리 위치 변경**
  - 툴 에러 처리가 기본 API가 아니라 **middleware의 `wrap_tool_call`** 쪽으로 이동했습니다.

### 구조화 출력 관련
- **Structured Output 방식 변경**
  - 예전의 “pro

## 7.5 LangChain 에이전트로 비교

같은 검색 도구를 LangChain `create_agent()`로도 사용해 봅니다.

**LangChain 에이전트와의 차이점:**

LangChain의 `create_agent()`는 모델과 도구를 받아 간단한 ReAct 에이전트를 생성합니다. Deep Agents와 비교하면:

- **LangChain**: 도구 호출 에이전트의 기본 형태. 빠른 프로토타이핑에 적합하지만, 태스크 플래닝이나 파일 시스템 관리 같은 고급 기능은 직접 구현해야 합니다.
- **Deep Agents**: 플래닝(`write_todos`), 파일 관리, 서브에이전트 위임이 기본 내장되어 있어, 복잡한 멀티스텝 작업에 더 적합합니다.

`@tool` 데코레이터를 사용하면 LangChain의 도구 인터페이스에 맞게 함수를 변환할 수 있습니다. 시스템 프롬프트와 도구 리스트를 `create_agent()`에 전달하는 패턴은 Deep Agents와 동일합니다.

In [10]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search_web(query: str) -> dict:
    """웹에서 정보를 검색합니다."""
    return tavily.search(query, max_results=3)

lc_agent = create_agent(
    model=model,
    tools=[search_web],
    system_prompt="당신은 리서치 어시스턴트입니다. 한국어로 답변하세요.",
)

result = lc_agent.invoke(
    {"messages": [{"role": "user", "content": "LangChain v1의 주요 특징을 검색해서 알려주세요."}]},
    config=lf_config,
)
print(result["messages"][-1].content)

검색 결과를 바탕으로 정리하면, **LangChain v1의 주요 특징**은 다음과 같습니다.

## 1) `create_agent` / `createAgent` 중심의 단순화된 에이전트 API
LangChain v1에서는 에이전트를 만드는 기본 진입점이 훨씬 단순해졌습니다.

- Python: `create_agent`
- JavaScript: `createAgent`

이전보다 더 간단한 인터페이스로 에이전트를 만들 수 있고,
내부적으로는 **모델 호출 → 도구 선택/실행 → 종료**의 기본 에이전트 루프를 사용합니다.

즉, **빠르게 시작하기 쉽고**, 동시에 아래의 미들웨어 구조를 통해 **고급 커스터마이징도 가능**해졌습니다.

---

## 2) 미들웨어(Middleware) 기반 확장성
v1의 핵심 변화 중 하나는 **미들웨어 중심 설계**입니다.

미들웨어를 통해 다음과 같은 로직을 주입할 수 있습니다.

- 모델 호출 전/후 처리
- 툴 호출 전/후 처리
- 동적 프롬프트 구성
- 컨텍스트 주입
- 모델 선택 전략 변경
- 에러 처리

공식 문서에서도 v1의 `create_agent`가 **highly customizable entry-point**라고 설명합니다.
즉, 단순한 체인 조합보다 **에이전트의 실행 흐름을 더 세밀하게 제어**할 수 있습니다.

---

## 3) LangGraph 기반 아키텍처
LangChain v1의 에이전트는 **LangGraph 위에 구축**됩니다.

이 의미는 다음과 같습니다.

- 더 신뢰성 있는 에이전트 실행
- 복잡한 흐름 제어 가능
- 장기 실행/상태 관리에 유리
- 프로덕션 환경에서 더 안정적인 구조

즉, LangChain은 상위 레벨의 쉬운 API를 제공하고,
그 아래에는 **LangGraph의 실행 엔진**이 받쳐주는 구조라고 볼 수 있습니다.

---

## 4) 구조화된 출력(Structured Output) 강화
v1에서는 **구조화된 출력 생성 기능이 강화**되었습니다.

특히 공식 

## 요약

이 미니 프로젝트에서 사용한 기술:

| 기술 | 출처 |
|---|---|
| `ChatOpenAI` + `load_dotenv` | 00_setup |
| 메시지 역할, 스트리밍 | 01_llm_basics |
| `@tool`, `create_agent()` | 02_langchain_basics |
| `InMemorySaver`, `thread_id` | 03_langchain_memory |
| `StateGraph`, `compile()` | 04_langgraph_basics |
| `create_deep_agent()` | 05_deep_agents_basics |

### 다음 단계
→ 중급 과정으로 진행하세요! **[06_comparison.ipynb](./06_comparison.ipynb)** 에서 안내를 확인하세요.
